<a href="https://colab.research.google.com/github/anjali062428-cmyk/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub pandas scikit-learn

In [4]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF token loaded:", bool(HF_TOKEN))

HF token loaded: True


In [5]:
import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

con.execute("SET VARIABLE hf_token = ?", [HF_TOKEN])

con.execute("""
    CREATE OR REPLACE SECRET hf
    (TYPE huggingface, TOKEN getvariable('hf_token'))
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT_MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
FACT_FEB = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-02/*.parquet')"

print("DuckDB connected successfully.")
print("Feature window: February 2026")
print("Verification/outcome window: March 2026")

DuckDB connected successfully.
Feature window: February 2026
Verification/outcome window: March 2026


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents one page/query record for a specific month in the Search Intelligence warehouse. I will use a mid-panel month, March 2026, for this assignment so that the final month remains a sealed test month.

One row represents one client-content-date observation in the Search Intelligence warehouse. The verification window is March 2026, covering 2026-03-01 through 2026-03-31. The March dataset contains 9,841,378 rows, with 0 duplicate client-content-date combinations. Data availability is explicitly represented by gsc_data_available; 3,611,061 rows are marked TRUE, while 6,230,317 rows are unavailable or NULL.

In [6]:
# Verify the raw daily grain for March 2026

query_1 = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (client_hash_id, content_hash_id, report_date))
            AS distinct_client_content_date_rows,
        COUNT(*)
        - COUNT(DISTINCT (client_hash_id, content_hash_id, report_date))
            AS duplicate_rows
    FROM {FACT_MAR}
""").df()

query_1

,total_rows,distinct_client_content_date_rows,duplicate_rows
0,9841378,9841378,0


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [7]:
# Verify row count and date span for March 2026

query_2 = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {FACT_MAR}
""").df()

query_2

,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [9]:
# Inspect the actual February 2026 warehouse columns

feb_sample = con.sql(f"""
    SELECT *
    FROM {FACT_FEB}
    LIMIT 1
""").df()

print("February columns:")
print(feb_sample.columns.tolist())

February columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [10]:
for i, col in enumerate(feb_sample.columns, start=1):
    print(i, col)

1 report_date
2 client_hash_id
3 content_hash_id
4 client_has_gsc
5 client_has_ga4
6 gsc_data_available
7 ga4_data_available
8 gsc_impressions
9 gsc_clicks
10 gsc_sum_position
11 gsc_avg_position
12 ga4_pageviews
13 ga4_sessions
14 ga4_users
15 ga4_engaged_sessions
16 ga4_total_engagement_sec
17 sessions_organic
18 sessions_direct
19 sessions_referral
20 sessions_social
21 sessions_paid
22 sessions_ai
23 ai_chatgpt
24 ai_perplexity
25 ai_gemini
26 ai_copilot
27 ai_claude
28 ai_meta
29 ai_other
30 scroll_events
31 month


In [12]:
# Build a small February feature frame efficiently

feb_features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions_feb,
        SUM(gsc_clicks) AS clicks_feb,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
            ELSE 0
        END AS ctr_feb,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_sum_position) * 1.0 / SUM(gsc_impressions)
            ELSE NULL
        END AS avg_position_feb,

        COUNT(*) AS observed_days_feb

    FROM (
        SELECT *
        FROM {FACT_FEB}
        WHERE gsc_data_available IS TRUE
        LIMIT 200000
    )

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

print("Feature frame shape:", feb_features.shape)
print("Five features:")
print([
    "impressions_feb",
    "clicks_feb",
    "ctr_feb",
    "avg_position_feb",
    "observed_days_feb"
])

feb_features.head()

Feature frame shape: (96472, 7)
Five features:
['impressions_feb', 'clicks_feb', 'ctr_feb', 'avg_position_feb', 'observed_days_feb']


,client_hash_id,content_hash_id,impressions_feb,clicks_feb,ctr_feb,avg_position_feb,observed_days_feb
0,client_23a62021009f63c4,content_597bda7763fabc08,9.0,0.0,0.000000,6.000000,3
1,client_23a62021009f63c4,content_4af30e56a906b288,759.0,9.0,0.011858,6.820817,3
2,client_23a62021009f63c4,content_399fc1c697a41bff,99.0,0.0,0.000000,1.878788,3
3,client_23a62021009f63c4,content_8fb9a5f52b37b2c8,116.0,0.0,0.000000,0.396552,3
4,client_23a62021009f63c4,content_83f83380c6f3e44e,348.0,1.0,0.002874,18.997126,3


Five features and why they are knowable at the decision moment:

1. impressions_feb — Knowable at the decision moment because these are aggregated GSC impressions observed during the February 2026 feature window.

2. clicks_feb — Knowable at the decision moment because these are aggregated GSC clicks observed during the February 2026 feature window.

3. ctr_feb — Knowable at the decision moment because it is calculated from February clicks and February impressions only.

4. avg_position_feb — Knowable at the decision moment because it is calculated from February GSC position measurements using impression-weighted aggregation.

5. observed_days_feb — Knowable at the decision moment because it counts the February observations where GSC data was explicitly available.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [8]:
# Verify that March 2026 availability is explicitly TRUE

query_3 = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS available_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS NOT TRUE) AS unavailable_or_null_rows
    FROM {FACT_MAR}
""").df()

query_3

,total_rows,available_rows,unavailable_or_null_rows
0,9841378,3611061,6230317


In [13]:
# Create the March 2026 outcome label: went_dark
# Label uses only measured March data.

march_label = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        CASE
            WHEN SUM(gsc_clicks) = 0 THEN 1
            ELSE 0
        END AS went_dark

    FROM {FACT_MAR}

    WHERE gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id

    HAVING SUM(gsc_impressions) >= 100
""").df()

print("March label frame shape:", march_label.shape)
print("\nLabel distribution:")
print(march_label["went_dark"].value_counts())

march_label.head()

March label frame shape: (101441, 3)

Label distribution:
went_dark
0    63662
1    37779
Name: count, dtype: int64


,client_hash_id,content_hash_id,went_dark
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,0
1,client_73cda7b4e4f265ea,content_905aa32a0230694e,1
2,client_73cda7b4e4f265ea,content_05434271b257bb68,0
3,client_73cda7b4e4f265ea,content_d056587ff7faca0c,0
4,client_73cda7b4e4f265ea,content_2662845f598544ef,0


In [14]:
# Deliberate leakage experiment
# We intentionally add the future March label to the feature frame.

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

leak_df = feb_features.merge(
    march_label,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

X_leak = leak_df[
    [
        "impressions_feb",
        "clicks_feb",
        "ctr_feb",
        "avg_position_feb",
        "observed_days_feb",
        "went_dark"          # DELIBERATE LEAK
    ]
]

y = leak_df["went_dark"]

X_train, X_test, y_train, y_test = train_test_split(
    X_leak,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

leak_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

leak_model.fit(X_train, y_train)

leak_pred = leak_model.predict(X_test)

leak_accuracy = accuracy_score(y_test, leak_pred)

print("Leaky feature used: went_dark")
print("Leakage-test accuracy:", round(leak_accuracy, 4))

Leaky feature used: went_dark
Leakage-test accuracy: 1.0


In [15]:
# Remove the leaked future label and evaluate only pre-decision features

X_clean = leak_df[
    [
        "impressions_feb",
        "clicks_feb",
        "ctr_feb",
        "avg_position_feb",
        "observed_days_feb"
    ]
]

y = leak_df["went_dark"]

X_train, X_test, y_train, y_test = train_test_split(
    X_clean,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

clean_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

clean_model.fit(X_train, y_train)

clean_pred = clean_model.predict(X_test)

clean_accuracy = accuracy_score(y_test, clean_pred)

print("Leaked feature removed: went_dark")
print("Clean-model accuracy:", round(clean_accuracy, 4))

Leaked feature removed: went_dark
Clean-model accuracy: 0.7635


Leakage check:

The March outcome label `went_dark` was deliberately added as a feature. The resulting accuracy was 1.0000, showing that the model had direct access to the answer it was supposed to predict.

This feature was then removed. Using only the five February features, the clean model accuracy was 0.7635.

This confirms that `went_dark` is a label-derived future field and must not be used as a model feature.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Data limitations:

This dataset cannot fully describe performance outside the observed Search Intelligence and GSC measurements. Unavailable GSC observations are not equivalent to zero traffic, historical coverage may be unbalanced, and the February feature window and March outcome window must remain separated to avoid temporal leakage. The feature demonstration also uses a limited sample of the February warehouse because the full aggregation was interrupted in the notebook environment.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


This data cannot establish causation or explain why a page's performance changed. Historical coverage may be unbalanced, some early observations may be GSC-only, and feature/outcome windows can overlap if the cutoff is not enforced carefully. Therefore, the data should be used for decision support rather than causal conclusions.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.